# Order-to-Fill Execution Quality Pipeline Exploration
This notebook demonstrates step-by-step data ingestion, data profiling, validation rule enforcement, relational workflow modeling, and metric computation for retail trading execution quality.

## 1. Raw Data Ingestion
We pull real market OHLC prices via `yfinance` and read simulated signal & order CSV logs.

In [ ]:
import os
import pandas as pd
import sys
sys.path.insert(0, '../src')

from ingest import run_ingestion
run_ingestion(raw_dir='../data/raw')

mkt_df = pd.read_csv('../data/raw/market_prices.csv')
sig_df = pd.read_csv('../data/raw/signals.csv')
ord_df = pd.read_csv('../data/raw/orders.csv')

print(f"Market Prices shape: {mkt_df.shape}")
print(f"Signals shape: {sig_df.shape}")
print(f"Orders shape: {ord_df.shape}")

## 2. Profiling & Data Quality Validation
We execute non-destructive business rule validation across price bands, timestamp monotonicity, quantities, orphan references, and market hours.

In [ ]:
from validate import validate_pipeline_data

clean_sig, clean_ord, val_report = validate_pipeline_data(sig_df, ord_df)
print(f"Violations found: {len(val_report)}")
val_report.head(10)

## 3. Workflow Modeling (SQLite Entity-Event Model)
Cleaned records are staged into an SQLite database with explicit primary/foreign keys, generating a unified transaction dataset.

In [ ]:
from model import build_workflow_model

model_df = build_workflow_model(clean_sig, clean_ord, db_path='../data/processed/execution_warehouse.db')
print(f"Modeled transactions: {len(model_df)}")
model_df.head()

## 4. Execution Quality Metrics & Final Evidence Table
We compute latency percentiles, adverse slippage (in bps), fill rates, rejection breakdown, and total INR slippage cost.

In [ ]:
from metrics import compute_execution_metrics

evidence_df = compute_execution_metrics(model_df)
evidence_df